# 02 - Model Training

In this notebook, we train a baseline convolutional neural network model for binary steel defect classification.

The goal is to classify steel surface images as:

- `0`: No Defect
- `1`: Defect

We use transfer learning with a pretrained MobileNetV2 model as the feature extraction backbone.

In [5]:
# Import standard libraries
from pathlib import Path

# Import data handling libraries
import pandas as pd
import numpy as np

# Import visualization libraries
import matplotlib.pyplot as plt

# Import machine learning utilities
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, ConfusionMatrixDisplay

## 1. Define Paths and Load Labels

In this section, we define the project paths and load the binary image-level labels created during the data exploration step.

In [6]:
# Define project root path
PROJECT_ROOT = Path("..")

# Define data directories
DATA_DIR = PROJECT_ROOT / "data"
RAW_DATA_DIR = DATA_DIR / "raw"

# Define image and label paths
TRAIN_IMAGES_DIR = RAW_DATA_DIR / "train_images"
LABELS_PATH = DATA_DIR / "binary_labels.csv"

# Check if paths exist
print("Train images directory exists:", TRAIN_IMAGES_DIR.exists())
print("Binary labels file exists:", LABELS_PATH.exists())

Train images directory exists: True
Binary labels file exists: True


In [7]:
# Load binary image-level labels
labels_df = pd.read_csv(LABELS_PATH)

# Display the first rows
labels_df.head()

,ImageId,HasDefect
0,0002cc93b.jpg,1
1,00031f466.jpg,0
2,000418bfc.jpg,0
3,000789191.jpg,0
4,0007a71bf.jpg,1


In [8]:
# Check the shape of the labels dataframe
print("Labels dataframe shape:", labels_df.shape)

# Check column names
print("Columns:", labels_df.columns.tolist())

# Check class distribution
print("\nClass distribution:")
print(labels_df["HasDefect"].value_counts())

# Check class distribution as percentages
print("\nClass distribution (%):")
print(labels_df["HasDefect"].value_counts(normalize=True) * 100)

Labels dataframe shape: (12568, 2)
Columns: ['ImageId', 'HasDefect']

Class distribution:
HasDefect
1    6666
0    5902
Name: count, dtype: int64

Class distribution (%):
HasDefect
1    53.039465
0    46.960535
Name: proportion, dtype: float64


In [9]:
# Create full image paths for each image
labels_df["ImagePath"] = labels_df["ImageId"].apply(lambda x: TRAIN_IMAGES_DIR / x)

# Display the first rows
labels_df.head()

,ImageId,HasDefect,ImagePath
0,0002cc93b.jpg,1,..\data\raw\train_images\0002cc93b.jpg
1,00031f466.jpg,0,..\data\raw\train_images\00031f466.jpg
2,000418bfc.jpg,0,..\data\raw\train_images\000418bfc.jpg
3,000789191.jpg,0,..\data\raw\train_images\000789191.jpg
4,0007a71bf.jpg,1,..\data\raw\train_images\0007a71bf.jpg


In [10]:
# Check if all image paths exist
labels_df["PathExists"] = labels_df["ImagePath"].apply(lambda x: x.exists())

# Count how many paths exist and how many do not
print(labels_df["PathExists"].value_counts())

# Display rows with missing image files, if any
labels_df[labels_df["PathExists"] == False].head()

PathExists
True    12568
Name: count, dtype: int64


,ImageId,HasDefect,ImagePath,PathExists


## 2. Train, Validation, and Test Split

In this section, we split the dataset into three subsets:

- **Training set**: used to train the model.
- **Validation set**: used during training to monitor performance and detect overfitting.
- **Test set**: kept separate and used only at the end for final evaluation.

We use a stratified split based on the `HasDefect` label. This ensures that the proportion of defect and no-defect images remains approximately the same in the training, validation, and test sets. This is important because it prevents one subset from having a different class balance than the original dataset, which could lead to misleading model evaluation.

In [11]:
# Split the data into training, validation, and test sets

# First split:
# 70% will be used for training
# 30% will be kept temporarily for validation + test
train_df, temp_df = train_test_split(
    labels_df,
    test_size=0.30,
    random_state=42,
    stratify=labels_df["HasDefect"]
)

# Second split:
# The temporary 30% is split equally into validation and test sets
# This gives 15% validation and 15% test overall
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    stratify=temp_df["HasDefect"]
)

# Display split sizes
print("Training set shape:", train_df.shape)
print("Validation set shape:", val_df.shape)
print("Test set shape:", test_df.shape)

# Check class distribution in each split
print("\nTraining class distribution (%):")
print(train_df["HasDefect"].value_counts(normalize=True) * 100)

print("\nValidation class distribution (%):")
print(val_df["HasDefect"].value_counts(normalize=True) * 100)

print("\nTest class distribution (%):")
print(test_df["HasDefect"].value_counts(normalize=True) * 100)

Training set shape: (8797, 4)
Validation set shape: (1885, 4)
Test set shape: (1886, 4)

Training class distribution (%):
HasDefect
1    53.040809
0    46.959191
Name: proportion, dtype: float64

Validation class distribution (%):
HasDefect
1    53.050398
0    46.949602
Name: proportion, dtype: float64

Test class distribution (%):
HasDefect
1    53.022269
0    46.977731
Name: proportion, dtype: float64


In [12]:
# Reset indexes after splitting
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

# Display the first rows of the training set
train_df.head()

,ImageId,HasDefect,ImagePath,PathExists
0,9c9ea5f57.jpg,1,..\data\raw\train_images\9c9ea5f57.jpg,True
1,ad9688790.jpg,0,..\data\raw\train_images\ad9688790.jpg,True
2,fcb25f2ec.jpg,1,..\data\raw\train_images\fcb25f2ec.jpg,True
3,63b63d495.jpg,1,..\data\raw\train_images\63b63d495.jpg,True
4,62b0a31ba.jpg,1,..\data\raw\train_images\62b0a31ba.jpg,True


## 3. Image Preprocessing and Data Pipeline

In this section, we define the image preprocessing settings that will be used before feeding the images into the CNN model.

The images are resized to a fixed square input size of `224 × 224` pixels to make them compatible with MobileNetV2. Since the original steel strip images are very wide, this resizing may distort the aspect ratio. This is acceptable for an initial binary classification baseline, but future improvements could include aspect-ratio-preserving preprocessing or patch-based classification.


In [13]:
# Define image preprocessing settings
IMG_SIZE = (224, 224)

# Define batch size
BATCH_SIZE = 32

# Display settings
print("Image size:", IMG_SIZE)
print("Batch size:", BATCH_SIZE)

Image size: (224, 224)
Batch size: 32


## 4. TensorFlow Dataset Pipeline

In this section, we create TensorFlow datasets for training, validation, and testing.

The pipeline reads image files from disk, decodes them as RGB images, resizes them to the selected input size, applies the MobileNetV2 preprocessing function, and batches the data for model training.

In [ ]:
# Import TensorFlow and Keras modules
import tensorflow as tf

# Import MobileNetV2 model and preprocessing function
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

# Import Keras layers and model utilities
from tensorflow.keras import layers, models

In [ ]:
# Define a function to load and preprocess a single image
def load_and_preprocess_image(image_path, label):
    # Read image file from disk
    image = tf.io.read_file(image_path)
    
    # Decode JPEG image and force 3 color channels (RGB)
    image = tf.image.decode_jpeg(image, channels=3)
    
    # Resize image to the selected input size
    image = tf.image.resize(image, IMG_SIZE)
    
    # Apply MobileNetV2-specific preprocessing
    image = preprocess_input(image)
    
    # Return preprocessed image and its label
    return image, label

In [ ]:
# Define a function to create a TensorFlow dataset from a dataframe
def create_dataset(df, shuffle=False):
    # Convert image paths to strings
    image_paths = df["ImagePath"].astype(str).values
    
    # Convert labels to integers
    labels = df["HasDefect"].astype(int).values
    
    # Create a TensorFlow dataset from image paths and labels
    dataset = tf.data.Dataset.from_tensor_slices((image_paths, labels))
    
    # Load and preprocess each image
    dataset = dataset.map(load_and_preprocess_image, num_parallel_calls=tf.data.AUTOTUNE)
    
    # Shuffle only the training dataset
    if shuffle:
        dataset = dataset.shuffle(buffer_size=len(df), seed=42)
    
    # Group images into batches
    dataset = dataset.batch(BATCH_SIZE)
    
    # Prefetch batches for better performance
    dataset = dataset.prefetch(tf.data.AUTOTUNE)
    
    return dataset

In [ ]:
# Create TensorFlow datasets
train_dataset = create_dataset(train_df, shuffle=True)
val_dataset = create_dataset(val_df, shuffle=False)
test_dataset = create_dataset(test_df, shuffle=False)

# Display dataset objects
print(train_dataset)
print(val_dataset)
print(test_dataset)

## 5. MobileNetV2 Model Definition

In this section, we define the baseline CNN model using transfer learning.

We use MobileNetV2 pretrained on ImageNet as the feature extraction backbone. The pretrained convolutional base is kept frozen during the initial training stage, and a custom binary classification head is added on top.

The final output uses a sigmoid activation function because this is a binary classification problem.

In [ ]:
# Load the pretrained MobileNetV2 model without the original classification head
base_model = MobileNetV2(
    weights="imagenet",
    include_top=False,
    input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3)
)

# Freeze the pretrained base model
base_model.trainable = False

# Build the binary classification model
model = models.Sequential([
    base_model,
    
    # Convert feature maps into a single feature vector
    layers.GlobalAveragePooling2D(),
    
    # Add dropout to reduce overfitting
    layers.Dropout(0.3),
    
    # Final binary classification output
    layers.Dense(1, activation="sigmoid")
])

# Display model architecture
model.summary()

In [ ]:
# Compile the model
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

## 6. Model Training

In this section, we train the MobileNetV2-based binary classification model.

During training, the model learns from the training dataset, while the validation dataset is used to monitor performance on unseen images and detect possible overfitting.

In [ ]:
# Train the model
EPOCHS = 10

history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=EPOCHS
)

## 7. Training Curves

In this section, we visualize the training and validation performance across epochs.

The accuracy and loss curves help us understand whether the model is learning properly and whether there are signs of overfitting.

In [ ]:
# Plot training and validation accuracy
plt.figure(figsize=(8, 5))

plt.plot(history.history["accuracy"], label="Training Accuracy")
plt.plot(history.history["val_accuracy"], label="Validation Accuracy")

plt.title("Training and Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Plot training and validation loss
plt.figure(figsize=(8, 5))

plt.plot(history.history["loss"], label="Training Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")

plt.title("Training and Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)
plt.show()

## 8. Test Set Evaluation

In this section, we evaluate the trained model on the held-out test set.

The test set was not used during training or validation, so it provides a more objective estimate of the model's final performance.

In [ ]:
# Generate predicted probabilities for the test set
y_pred_prob = model.predict(test_dataset)

# Convert probabilities to binary predictions using a 0.5 threshold
y_pred = (y_pred_prob >= 0.5).astype(int).flatten()

# Extract true labels from the test dataframe
y_true = test_df["HasDefect"].astype(int).values

# Display first predictions
print("First predicted probabilities:")
print(y_pred_prob[:10].flatten())

print("\nFirst binary predictions:")
print(y_pred[:10])

print("\nFirst true labels:")
print(y_true[:10])

In [ ]:
# Calculate evaluation metrics
test_accuracy = accuracy_score(y_true, y_pred)
test_precision = precision_score(y_true, y_pred)
test_recall = recall_score(y_true, y_pred)
test_f1 = f1_score(y_true, y_pred)

# Display evaluation metrics
print("Test Accuracy:", test_accuracy)
print("Test Precision:", test_precision)
print("Test Recall:", test_recall)
print("Test F1-score:", test_f1)

In [ ]:
# Compute confusion matrix
cm = confusion_matrix(y_true, y_pred)

# Display confusion matrix
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["No Defect", "Defect"]
)

disp.plot(values_format="d")
plt.title("Confusion Matrix - Test Set")
plt.show()

## 9. Save Results

In this section, we save the main training and evaluation figures to the `results/` folder so they can be used later in the project README.

In [15]:
# Define results directory
RESULTS_DIR = PROJECT_ROOT / "results"

# Create results directory if it does not already exist
RESULTS_DIR.mkdir(exist_ok=True)

# Check results directory
print("Results directory exists:", RESULTS_DIR.exists())

Results directory exists: True


In [ ]:
# Save training and validation accuracy curve
plt.figure(figsize=(8, 5))

plt.plot(history.history["accuracy"], label="Training Accuracy")
plt.plot(history.history["val_accuracy"], label="Validation Accuracy")

plt.title("Training and Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(True)

plt.savefig(RESULTS_DIR / "training_accuracy.png", dpi=300, bbox_inches="tight")
plt.show()


# Save training and validation loss curve
plt.figure(figsize=(8, 5))

plt.plot(history.history["loss"], label="Training Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")

plt.title("Training and Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)

plt.savefig(RESULTS_DIR / "training_loss.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# Save confusion matrix figure
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["No Defect", "Defect"]
)

disp.plot(values_format="d")
plt.title("Confusion Matrix - Test Set")

plt.savefig(RESULTS_DIR / "confusion_matrix.png", dpi=300, bbox_inches="tight")
plt.show()

## 10. Save Trained Model

In this section, we save the trained MobileNetV2-based model so it can be reused later for explainability analysis with Grad-CAM.

In [16]:
# Define model output directory
MODEL_DIR = PROJECT_ROOT / "models"

# Create model directory if it does not already exist
MODEL_DIR.mkdir(exist_ok=True)

# Check model directory
print("Model directory exists:", MODEL_DIR.exists())

Model directory exists: True


In [ ]:
# Save the trained model
model.save(MODEL_DIR / "mobilenetv2_baseline.keras")

## Summary

In this notebook, we prepared and defined a baseline transfer learning pipeline for binary steel defect classification.

The dataset was split into training, validation, and test subsets using a stratified 70/15/15 split to preserve the original defect/no-defect class distribution.

A MobileNetV2 model pretrained on ImageNet was selected as the baseline CNN backbone. The model uses a custom binary classification head with a sigmoid output for predicting whether a steel surface image contains a defect.

The images are resized to `224 × 224` pixels for compatibility with MobileNetV2. Since the original steel strip images are very wide, this square resizing may distort the aspect ratio and may reduce sensitivity to small or thin defects. Future improvements could include aspect-ratio-preserving preprocessing or patch-based classification.

The trained model and evaluation figures are saved for use in later stages of the project, including Grad-CAM explainability analysis.